# 03 — Analyse des associations de produits

**Objectif :** construire la matrice d’association puis calculer le *support*, la *confiance*, la *confiance attendue* et le *lift*, conformément à **Cours associations**, pages 6 à 8.

**Population principale :** segment initial, choisi et justifié dans le notebook 02. **Comparaison :** base complète agrégée par identifiant. Les résultats ne prouvent pas qu’une association entraîne un achat.

## 1. Chargement des données et des fonctions

Les calculs sont regroupés dans `src/associations.py` ; le notebook présente les observations et les interprétations.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

racine = Path.cwd()
if not (racine / 'src').is_dir():
    racine = racine.parent
if not (racine / 'src').is_dir():
    raise FileNotFoundError('Exécuter depuis la racine du projet ou notebooks/.')
if str(racine) not in sys.path:
    sys.path.insert(0, str(racine))

from src.chargement import charger_donnees, colonnes_produits
from src.preparation import base_initiale, base_agregee
from src.associations import (
    matrice_association, indicateurs_association, comparer_indicateurs
)

df = charger_donnees(racine)
produits = colonnes_produits(df)
matrice_initiale = base_initiale(df, produits)
matrice_agregee = base_agregee(df, produits)

print(f'Base principale : {len(matrice_initiale):,} clients, {len(produits)} produits')
print(f'Base complémentaire : {len(matrice_agregee):,} clients')

Base principale : 72,531 clients, 26 produits
Base complémentaire : 72,638 clients


## 2. Construction de la matrice d’association

Chaque case contient le nombre de clients détenant **les deux produits**. La diagonale donne le nombre de clients détenant chaque produit. Voir l’exemple de la page 7 du cours.

In [2]:
effectifs = matrice_association(matrice_initiale)
display(effectifs)

# Quelques contrôles de cohérence
assert (effectifs.to_numpy() == effectifs.to_numpy().T).all()
assert (effectifs.to_numpy() <= len(matrice_initiale)).all()
print('Matrice carrée et symétrique :', effectifs.shape)

produit_2,CART,CJEU,PACKSERV,DAV,DECAUTH,CPAI,CARTE,auto,GAV,PJ,...,ASSMH,CREDHAB,CSL,OBLIGATION,SICAVFCP,EPARBANC,ASSPREVDEC,ASSDEP,CREDRENOV,ASSPREVDECDEP
produit_2,,,,,,,,,,,,,,,,,,,,,
CART,61638,8027,53908,61529,41980,57889,1146,20963,6617,26471,...,39453,18686,53889,5076,8460,4058,10167,1363,7803,11270
CJEU,8027,8027,7380,7981,5099,5286,117,2995,827,3302,...,4283,1667,7640,209,735,211,2580,93,1007,2613
PACKSERV,53908,7380,60031,60031,45852,50842,1120,19966,6706,27233,...,37948,17909,52281,5242,8308,4018,10083,1503,8054,11319
DAV,61529,7981,60031,71834,46483,57830,1154,23699,7581,30582,...,46186,20883,60775,6574,10132,5068,10861,1630,8537,12210
DECAUTH,41980,5099,45852,46483,46483,40544,1024,15949,5097,20440,...,29838,14695,40626,3548,5820,2875,7246,1007,6785,8033
CPAI,57889,5286,50842,57830,40544,57889,1040,19956,6384,24894,...,37484,18339,50639,4957,8196,3955,8852,1311,7476,9919
CARTE,1146,117,1120,1154,1024,1040,1154,517,177,650,...,678,377,995,36,92,45,256,65,705,304
auto,20963,2995,19966,23699,15949,19956,517,23894,3484,8728,...,13924,6044,20799,2289,3510,1850,3838,652,3186,4363
GAV,6617,827,6706,7581,5097,6384,177,3484,7591,5375,...,5682,2109,6996,1794,2957,1271,1152,748,1471,1781


Matrice carrée et symétrique : (26, 26)


## 3. Indicateurs des associations orientées A → B

- **Support** : clients détenant A et B / clients étudiés.
- **Confiance** : clients détenant A et B / clients détenant A.
- **Confiance attendue** : clients détenant B / clients étudiés.
- **Lift** : confiance / confiance attendue.

Les ratios sont stockés entre 0 et 1 (sauf le lift) et affichés en pourcentage ci-dessous. Les associations A → B et B → A sont distinguées.

In [3]:
regles_initiales = indicateurs_association(matrice_initiale)

colonnes = ['produit_a', 'produit_b', 'nb_clients_a_et_b',
            'support', 'confiance', 'confiance_attendue', 'lift']
aperçu = regles_initiales[colonnes].copy()
for colonne in ['support', 'confiance', 'confiance_attendue']:
    aperçu[colonne] = (aperçu[colonne] * 100).round(2)
aperçu['lift'] = aperçu['lift'].round(3)
print('Nombre d’associations orientées :', len(aperçu))
display(aperçu.head(15))

Nombre d’associations orientées : 650


,produit_a,produit_b,nb_clients_a_et_b,support,confiance,confiance_attendue,lift
0,CART,CJEU,8027,11.07,13.02,11.07,1.177
1,CART,PACKSERV,53908,74.32,87.46,82.77,1.057
2,CART,DAV,61529,84.83,99.82,99.04,1.008
3,CART,DECAUTH,41980,57.88,68.11,64.09,1.063
4,CART,CPAI,57889,79.81,93.92,79.81,1.177
5,CART,CARTE,1146,1.58,1.86,1.59,1.169
6,CART,auto,20963,28.90,34.01,32.94,1.032
7,CART,GAV,6617,9.12,10.74,10.47,1.026
8,CART,PJ,26471,36.50,42.95,42.44,1.012
9,CART,Sante,7469,10.30,12.12,11.39,1.064


## 4. Lecture d’une association (exemple)

On choisit une association identifiable dans les données pour expliquer les indicateurs. Il ne s’agit pas encore d’une recommandation commerciale.

In [4]:
produit_a = 'DAV'
produit_b = 'ASSMH'

exemple = regles_initiales.loc[
    (regles_initiales['produit_a'] == produit_a) &
    (regles_initiales['produit_b'] == produit_b)
]
if exemple.empty:
    print('Association indisponible : vérifier les colonnes du fichier.')
else:
    r = exemple.iloc[0]
    print(f'Association : {produit_a} → {produit_b}')
    print(f'Support : {r.support:.2%}')
    print(f'Confiance : {r.confiance:.2%}')
    print(f'Confiance attendue : {r.confiance_attendue:.2%}')
    print(f'Lift : {r.lift:.3f}')

Association : DAV → ASSMH
Support : 63.68%
Confiance : 64.30%
Confiance attendue : 64.23%
Lift : 1.001


## 5. Comparaison entre les deux populations

Les différences sont calculées pour les mêmes associations dans les deux bases. La proximité des fréquences individuelles, observée au notebook 02, ne garantit pas à elle seule la proximité des associations.

In [5]:
regles_agregees = indicateurs_association(matrice_agregee)
comparaison = comparer_indicateurs(regles_initiales, regles_agregees)

# Comparer d'abord les associations avec le plus grand écart absolu de support.
comparaison_support = comparaison.assign(
    ecart_abs_support_pp = comparaison['ecart_support'].abs() * 100
).sort_values('ecart_abs_support_pp', ascending=False)

display(comparaison_support[[
    'produit_a', 'produit_b',
    'support_initial', 'support_agregation', 'ecart_abs_support_pp',
    'confiance_initial', 'confiance_agregation',
    'lift_initial', 'lift_agregation'
]].head(15).round(4))

,produit_a,produit_b,support_initial,support_agregation,ecart_abs_support_pp,confiance_initial,confiance_agregation,lift_initial,lift_agregation
481,OBLIGATION,CARTE,0.0005,0.0021,0.1638,0.0055,0.0231,0.3437,1.3199
168,CARTE,OBLIGATION,0.0005,0.0021,0.1638,0.0312,0.1218,0.3437,1.3199
170,CARTE,EPARBANC,0.0006,0.0023,0.1637,0.0390,0.1288,0.5557,1.7965
531,EPARBANC,CARTE,0.0006,0.0023,0.1637,0.0088,0.0315,0.5557,1.7965
157,CARTE,GAV,0.0024,0.0041,0.1635,0.1534,0.2325,1.4655,2.1906
206,GAV,CARTE,0.0024,0.0041,0.1635,0.0233,0.0384,1.4655,2.1906
161,CARTE,ACTION,0.0037,0.0054,0.1633,0.2340,0.3056,1.0694,1.3888
306,ACTION,CARTE,0.0037,0.0054,0.1633,0.0170,0.0243,1.0694,1.3888
181,auto,CARTE,0.0071,0.0088,0.1628,0.0216,0.0265,1.3599,1.5115
156,CARTE,auto,0.0071,0.0088,0.1628,0.4480,0.4996,1.3599,1.5115


## 6. Premières pistes : assurance et épargne

Le sujet privilégie les produits d’assurance et d’épargne. Pour commencer, on peut explorer des conséquences **à titre d’exemple** (`ASSMH` et `EPARBANC`) ; il faudra confirmer l’ensemble des familles de produits à partir du dictionnaire de données.

Les règles ne sont pas encore retenues : le cours demande d’étudier **ensemble** support, confiance et lift, puis de justifier les seuils choisis (pages 8–9).

In [6]:
produits_cibles_exemple = ['ASSMH', 'EPARBANC']
produits_cibles_exemple = [p for p in produits_cibles_exemple if p in produits]

regles_cibles = regles_initiales[
    regles_initiales['produit_b'].isin(produits_cibles_exemple)
].copy()

# Tri descriptif seulement : pas un seuil de sélection définitif.
display(regles_cibles.sort_values(
    ['produit_b', 'support'], ascending=[True, False]
)[['produit_a', 'produit_b', 'support', 'confiance',
   'confiance_attendue', 'lift']].round(4))

,produit_a,produit_b,support,confiance,confiance_attendue,lift
90,DAV,ASSMH,0.6368,0.6430,0.6423,1.0010
15,CART,ASSMH,0.5439,0.6401,0.6423,0.9966
466,CSL,ASSMH,0.5416,0.6429,0.6423,1.0010
65,PACKSERV,ASSMH,0.5232,0.6321,0.6423,0.9842
140,CPAI,ASSMH,0.5168,0.6475,0.6423,1.0081
115,DECAUTH,ASSMH,0.4114,0.6419,0.6423,0.9994
365,CREDCONSO,ASSMH,0.2703,0.6102,0.6423,0.9500
390,EPARLOG,ASSMH,0.2513,0.6801,0.6423,1.0589
240,PJ,ASSMH,0.2335,0.5502,0.6423,0.8567
441,CREDHAB,ASSMH,0.2298,0.7974,0.6423,1.2415


## 7. Interprétation et suite du projet

**À compléter après exécution :**

1. Quelles associations ont un support suffisamment important ?
2. Quelles associations présentent une confiance élevée et un lift supérieur à 1, suffisamment éloigné de 1 ?
3. Les mêmes associations restent-elles pertinentes dans la base agrégée ?
4. Quelles associations concernent réellement l’assurance ou l’épargne et peuvent étayer des propositions de cross-selling ?

**Étape suivante :** choisir et justifier les seuils selon le sujet, puis formuler des préconisations commerciales. Aucune conclusion ne doit être tirée sans examiner les résultats exécutés.